<a href="https://colab.research.google.com/github/mdM4yday/week7/blob/main/AIML202_Lab_DecisionTree_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIML202 - Machine Learning: Theory and Applications

## Lab: Training and Visualizing a Decision Tree Classifier

**Topic:** Attribute selection, tree depth and pruning, and their effect on accuracy and interpretability

**Duration:** 2 hours

| Part | Time | What happens |
|---|---|---|
| Part A - Guided walkthrough | about 60 minutes | The instructor goes through each section with you. Run every cell and read the explanations. |
| Part B - Graded exercises | about 30 minutes | You work on your own. 3 exercises, 100 marks in total. |

### Learning outcomes

By the end of this lab you should be able to:

1. Explain how a decision tree makes a prediction.
2. Calculate Gini impurity, entropy and information gain, and use them to choose the best attribute for a split.
3. Train and draw a decision tree with scikit-learn.
4. Explain how tree depth affects overfitting and underfitting.
5. Use pre-pruning to make a tree simpler, and judge the trade-off between accuracy and interpretability.

---
# PART A - Guided Walkthrough

## 1. Setup

Run the cell below first. It loads all the libraries we need today.

- `pandas` - for tables of data
- `numpy` - for maths
- `matplotlib` - for plots
- `sklearn` (scikit-learn) - for the decision tree and the datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score

print("All libraries loaded.")

## 2. What is a Decision Tree?

A decision tree is like a game of **"20 Questions"**. To make a prediction, the tree asks a series of simple yes/no questions about the data, one after another, until it reaches an answer.

Example: a doctor deciding if a patient has the flu.

```
                 Temperature > 38?
                 /               \
               Yes                No
               /                    \
        Has cough?               Not flu
        /        \
      Yes         No
      /             \
    Flu          Not flu
```

### Key words

| Word | Meaning |
|---|---|
| **Root node** | The first question at the top of the tree |
| **Internal node** | Any question in the middle of the tree |
| **Branch** | An answer to a question (Yes / No), it leads to the next node |
| **Leaf node** | The end of a path. It gives the final prediction (the class) |
| **Depth** | The number of questions from the root to the deepest leaf |

### The big question for today

When we build a tree, **which question should we ask first?** And **how many questions should we allow?**

- Choosing the question is called **attribute selection** (Section 3).
- Controlling the number of questions is done with **tree depth and pruning** (Sections 5 to 8).

## 3. Attribute Selection: Which Question Should Come First?

A good question splits the data into groups that are **pure**. A pure group contains only one class (for example, all "Passed" or all "Failed").

To measure how mixed (impure) a group is, we use **Gini impurity** or **Entropy**.

### 3.1 Gini impurity

$$\text{Gini} = 1 - (p_{yes}^2 + p_{no}^2)$$

where $p_{yes}$ is the fraction of "Yes" in the group and $p_{no}$ is the fraction of "No".

- Gini = 0 means the group is **perfectly pure** (only one class).
- Gini = 0.5 means the group is **completely mixed** (50% / 50%) for two classes.

### 3.2 Entropy

$$\text{Entropy} = -\,p_{yes}\log_2(p_{yes}) - p_{no}\log_2(p_{no})$$

- Entropy = 0 means **perfectly pure**.
- Entropy = 1 means **completely mixed** (50% / 50%) for two classes.

Both measure the same idea: **how mixed the group is**. Lower is better.

### 3.3 Information Gain

After a split we have several smaller groups. We take the **weighted average** impurity of those groups (bigger groups count more). Then:

$$\text{Gain} = \text{Impurity(before split)} - \text{Weighted impurity(after split)}$$

**The attribute with the highest gain is chosen for the split.** The tree repeats this at every node.

### 3.4 A small example: Will a student pass the exam?

We have data about 12 students. We want to predict `Passed` using three attributes: `Studied`, `Attendance` and `Slept_Well`.

In [ ]:
students = pd.DataFrame({
    "Studied":    ["Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "No",   "No",  "No",  "No",   "No",  "No"],
    "Attendance": ["High", "High", "Low", "Low", "Low", "High", "High", "Low", "Low", "High", "Low", "Low"],
    "Slept_Well": ["Yes", "No",  "Yes", "No",  "Yes", "Yes", "Yes",  "No",  "Yes", "No",   "No",  "Yes"],
    "Passed":     ["Yes", "Yes", "Yes", "Yes", "No",  "Yes", "Yes",  "No",  "No",  "No",   "No",  "No"],
})

students

Now we write two small functions. Each one takes the number of "Yes" and "No" in a group and returns the impurity.

In [ ]:
def gini(yes, no):
    total = yes + no
    p_yes = yes / total
    p_no = no / total
    return 1 - (p_yes ** 2 + p_no ** 2)


def entropy(yes, no):
    total = yes + no
    result = 0
    for count in [yes, no]:
        p = count / total
        if p > 0:                  # log2(0) is not defined, so we skip it
            result = result - p * np.log2(p)
    return result


# Quick test
print("Pure group (10 Yes, 0 No):   Gini =", gini(10, 0), " Entropy =", entropy(10, 0))
print("Mixed group (5 Yes, 5 No):   Gini =", gini(5, 5), " Entropy =", entropy(5, 5))

**Impurity before any split.** All 12 students together: let us count how many passed.

In [ ]:
yes_total = (students["Passed"] == "Yes").sum()
no_total = (students["Passed"] == "No").sum()

print("Passed = Yes:", yes_total)
print("Passed = No: ", no_total)
print("Gini before split:   ", round(gini(yes_total, no_total), 3))
print("Entropy before split:", round(entropy(yes_total, no_total), 3))

**Impurity after splitting on one attribute.** The function below splits the students by an attribute (for example `Studied`), calculates the impurity of each group, and returns the weighted average.

We go through it slowly:

1. Find the possible values of the attribute (for example "Yes" and "No").
2. For each value, take only the students with that value.
3. Count Yes and No in that group and compute its impurity.
4. Multiply by the group's share of students (its weight) and add it up.

In [ ]:
def impurity_after_split(data, attribute, measure, show=False):
    total_students = len(data)
    weighted_impurity = 0

    for value in data[attribute].unique():
        group = data[data[attribute] == value]
        yes = (group["Passed"] == "Yes").sum()
        no = (group["Passed"] == "No").sum()

        weight = len(group) / total_students
        weighted_impurity = weighted_impurity + weight * measure(yes, no)

        if show:
            print(f"   {attribute} = {value:5s}: {yes} Yes, {no} No, impurity = {measure(yes, no):.3f}")

    return weighted_impurity


# Example for one attribute
after = impurity_after_split(students, "Studied", gini, show=True)
print("Weighted Gini after split on Studied:", round(after, 3))

Now we calculate the **information gain** for every attribute and compare them.

In [ ]:
gini_before = gini(yes_total, no_total)
entropy_before = entropy(yes_total, no_total)

results = []
for attribute in ["Studied", "Attendance", "Slept_Well"]:
    gini_gain = gini_before - impurity_after_split(students, attribute, gini)
    entropy_gain = entropy_before - impurity_after_split(students, attribute, entropy)
    results.append([attribute, round(gini_gain, 3), round(entropy_gain, 3)])

gain_table = pd.DataFrame(results, columns=["Attribute", "Gini gain", "Information gain (entropy)"])
gain_table

**What do we see?**

- `Studied` has the **highest gain** with both measures, so it is the best first question (the root).
- `Slept_Well` has almost **zero gain**: knowing it barely helps us predict the result.
- Gini and entropy give different numbers, but they usually **rank the attributes in the same order**.

### 3.5 Let scikit-learn do it

scikit-learn does exactly this calculation for us. The model needs numbers, so we first change "Yes"/"High" to 1 and "No"/"Low" to 0.

In [ ]:
mapping = {"Yes": 1, "No": 0, "High": 1, "Low": 0}

X_students = pd.DataFrame()
X_students["Studied"] = students["Studied"].map(mapping)
X_students["Attendance"] = students["Attendance"].map(mapping)
X_students["Slept_Well"] = students["Slept_Well"].map(mapping)
y_students = students["Passed"]

small_tree = DecisionTreeClassifier(criterion="gini", random_state=0)
small_tree.fit(X_students, y_students)

plt.figure(figsize=(10, 6))
plot_tree(small_tree,
          feature_names=["Studied", "Attendance", "Slept_Well"],
          class_names=small_tree.classes_,
          filled=True, rounded=True)
plt.show()

**How to read each box in the tree:**

- **First line** - the question, for example `Studied <= 0.5` means "Studied = No?". If **True**, go **left**; if **False**, go **right**.
- **gini** - impurity of the students in this box.
- **samples** - how many students reach this box.
- **value** - how many students of each class `[No, Yes]`.
- **class** - the prediction if we stopped here (the majority class).
- **Colour** - the more intense the colour, the purer the box.

The root question is `Studied`, the same answer we calculated by hand.

## 4. A Real Dataset: Breast Cancer Diagnosis

Now we use a real medical dataset that comes with scikit-learn.

- **569 patients**, **30 numeric features** measured from images of cells (for example `mean radius`, `worst texture`).
- **Target:** `malignant` (cancer) or `benign` (not cancer).

### Train, validation and test sets

We split the data into **three** parts:

| Set | Share | Used for |
|---|---|---|
| **Training set** | 60% | The tree learns from this data |
| **Validation set** | 20% | We use it to **choose settings** such as depth (like a practice exam) |
| **Test set** | 20% | Used **only once at the end** to report the final accuracy (the real exam) |

If we chose settings using the test set, our final score would be too optimistic, because we would have "seen the exam questions" in advance.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target                      # 0 = malignant, 1 = benign
class_names = data.target_names.tolist()

print("Number of patients:", X.shape[0])
print("Number of features:", X.shape[1])
print("Classes:", class_names)
X.head()

In [ ]:
# Step 1: take 20% for the final test set
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=14, stratify=y)

# Step 2: split the remaining 80% into training (60%) and validation (20%)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=14, stratify=y_temp)

print("Training set:  ", X_train.shape[0], "patients")
print("Validation set:", X_val.shape[0], "patients")
print("Test set:      ", X_test.shape[0], "patients")

`stratify=y` keeps the same proportion of malignant and benign patients in every set. `random_state=14` makes the split the same every time we run the cell.

## 5. A Fully Grown Tree (No Limits)

By default, scikit-learn keeps splitting until **every leaf is pure**. Let us see what happens.

In [ ]:
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train)

train_acc = accuracy_score(y_train, full_tree.predict(X_train))
val_acc = accuracy_score(y_val, full_tree.predict(X_val))

print("Training accuracy:  ", round(train_acc, 3))
print("Validation accuracy:", round(val_acc, 3))
print("Tree depth:         ", full_tree.get_depth())
print("Number of leaves:   ", full_tree.get_n_leaves())

**What do we see?**

- Training accuracy is **1.0 (100%)**. The tree has memorised the training data.
- Validation accuracy is **lower**. The tree does not do as well on new patients.

This gap is called **overfitting**. A tree with many leaves can learn the noise in the training data, not only the real pattern.

### 5.1 Visualizing the tree

A full tree is often too large to read, so we draw only the **top 2 levels** using `max_depth=2` inside `plot_tree`. (This only limits the drawing, not the model.)

In [ ]:
plt.figure(figsize=(16, 7))
plot_tree(full_tree,
          max_depth=2,
          feature_names=list(X.columns),
          class_names=class_names,
          filled=True, rounded=True, fontsize=9)
plt.show()

We can also print the tree as **text rules**. This is useful when the drawing is too big. Here are the first 3 levels:

In [ ]:
rules = export_text(full_tree, feature_names=list(X.columns), max_depth=3)
print(rules)

Each line is one question. Lines ending in `class: ...` are leaves. Read it from top to bottom like an `if / else` program.

## 6. Effect of Tree Depth

`max_depth` limits how many questions the tree may ask in a row.

- **Too shallow** (depth 1 or 2): the tree is too simple and misses patterns. This is **underfitting**.
- **Too deep**: the tree memorises the training data. This is **overfitting**.
- **Just right**: good accuracy on new data.

Let us train trees with depth 1 to 12 and record the accuracy each time.

In [ ]:
depths = range(1, 13)
train_scores = []
val_scores = []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train, y_train)

    train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    val_scores.append(accuracy_score(y_val, model.predict(X_val)))

depth_table = pd.DataFrame({
    "max_depth": list(depths),
    "Train accuracy": np.round(train_scores, 3),
    "Validation accuracy": np.round(val_scores, 3),
})
depth_table

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(depths, train_scores, marker="o", label="Training accuracy")
plt.plot(depths, val_scores, marker="o", label="Validation accuracy")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Effect of tree depth on accuracy")
plt.xticks(list(depths))
plt.legend()
plt.grid(True)
plt.show()

**How to read this plot:**

- The **training** line keeps going up until it reaches 1.0. A deeper tree always fits the training data better.
- The **validation** line goes up at first, then stops improving or goes down. That is where overfitting starts.
- We choose the depth with the **best validation accuracy**. If two depths are equal, we choose the **smaller** one, because a smaller tree is easier to understand.

In [ ]:
best_index = int(np.argmax(val_scores))          # position of the highest validation score
best_depth = list(depths)[best_index]

print("Best max_depth:", best_depth)
print("Validation accuracy at this depth:", round(val_scores[best_index], 3))

`np.argmax` returns the **first** position of the highest value, so if there is a tie it already picks the smaller depth.

## 7. Pre-pruning: Stopping the Tree Early

**Pruning** means making a tree smaller and simpler, just like a gardener cuts branches from a real tree.

In this lab we use **pre-pruning** (also called early stopping): we give the tree rules that **stop it from growing too big while it is being built**. You already used one of these rules: `max_depth`.

| Setting | Meaning |
|---|---|
| `max_depth` | Maximum number of questions from root to leaf |
| `min_samples_split` | A node must have at least this many samples to be split further |
| `min_samples_leaf` | Every leaf must have at least this many samples |
| `max_leaf_nodes` | Maximum number of leaves in the whole tree |

Why does this help? A leaf with only 1 or 2 patients is usually learning a **special case** (noise), not a general rule. Asking each leaf to contain more samples forces the tree to learn rules that are true for many patients.

### 7.1 Comparing pre-pruning settings

In [ ]:
settings = [
    {"name": "No limits (full tree)",  "params": {}},
    {"name": "max_depth = 3",          "params": {"max_depth": 3}},
    {"name": "min_samples_leaf = 10",  "params": {"min_samples_leaf": 10}},
    {"name": "min_samples_split = 30", "params": {"min_samples_split": 30}},
    {"name": "max_leaf_nodes = 6",     "params": {"max_leaf_nodes": 6}},
]

rows = []
for s in settings:
    model = DecisionTreeClassifier(random_state=42, **s["params"])
    model.fit(X_train, y_train)
    rows.append([
        s["name"],
        round(accuracy_score(y_train, model.predict(X_train)), 3),
        round(accuracy_score(y_val, model.predict(X_val)), 3),
        model.get_depth(),
        model.get_n_leaves(),
    ])

pre_pruning_table = pd.DataFrame(rows, columns=["Setting", "Train acc", "Validation acc", "Depth", "Leaves"])
pre_pruning_table

**What do we see?**

- Every pre-pruning setting makes the tree **much smaller** (fewer leaves).
- Training accuracy goes **down** (the tree memorises less).
- Validation accuracy changes only **a little**: some settings are slightly better, some slightly worse. We get a much simpler tree without a big loss.

Note: `**s["params"]` simply passes the settings in the dictionary into the model, for example `max_depth=3`.

### 7.2 Choosing a good value for `min_samples_leaf`

Just like with `max_depth`, we try several values and look at the validation accuracy.

In [ ]:
leaf_values = [1, 2, 5, 10, 15, 20, 30]
leaf_train_scores = []
leaf_val_scores = []
leaf_n_leaves = []

for v in leaf_values:
    model = DecisionTreeClassifier(min_samples_leaf=v, random_state=42)
    model.fit(X_train, y_train)

    leaf_train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    leaf_val_scores.append(accuracy_score(y_val, model.predict(X_val)))
    leaf_n_leaves.append(model.get_n_leaves())

leaf_table = pd.DataFrame({
    "min_samples_leaf": leaf_values,
    "Train acc": np.round(leaf_train_scores, 3),
    "Validation acc": np.round(leaf_val_scores, 3),
    "Leaves": leaf_n_leaves,
})
leaf_table

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(leaf_values, leaf_train_scores, marker="o", label="Training accuracy")
plt.plot(leaf_values, leaf_val_scores, marker="o", label="Validation accuracy")
plt.xlabel("min_samples_leaf")
plt.ylabel("Accuracy")
plt.title("Effect of min_samples_leaf on accuracy")
plt.legend()
plt.grid(True)
plt.show()

**How to read this:**

- `min_samples_leaf = 1` is the full tree: training accuracy 1.0, many leaves.
- As the value grows, training accuracy goes **down** and the tree gets **smaller**, but validation accuracy stays the same or improves a little.
- If the value is too large, the tree becomes too simple and validation accuracy starts to drop (underfitting).

We choose the value with the best validation accuracy. If several values tie, we take the **largest** one, because it gives the **smallest** tree with the same accuracy.

In [ ]:
best_val = max(leaf_val_scores)

best_leaf = 1
for i in range(len(leaf_values)):
    if leaf_val_scores[i] == best_val:
        best_leaf = leaf_values[i]       # keeps the last (largest) value with the best score

print("Best validation accuracy:", round(best_val, 3))
print("Chosen min_samples_leaf:", best_leaf)

## 8. Accuracy vs Interpretability

Now let us compare three final models on the **test set**. This is the first and only time we use the test set.

1. The full tree (no limits)
2. The tree with the best `max_depth` (pre-pruning)
3. The tree with the best `min_samples_leaf` (pre-pruning)

In [ ]:
depth_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
depth_tree.fit(X_train, y_train)

pruned_tree = DecisionTreeClassifier(min_samples_leaf=best_leaf, random_state=42)
pruned_tree.fit(X_train, y_train)

final_models = [
    ["Full tree", full_tree],
    ["Best max_depth", depth_tree],
    ["Best min_samples_leaf", pruned_tree],
]

rows = []
for name, model in final_models:
    rows.append([
        name,
        round(accuracy_score(y_test, model.predict(X_test)), 3),
        model.get_depth(),
        model.get_n_leaves(),
    ])

comparison = pd.DataFrame(rows, columns=["Model", "Test accuracy", "Depth", "Leaves"])
comparison

**What do we see?**

- The pruned trees have **far fewer leaves** than the full tree.
- Their test accuracy is **higher** than the full tree. Removing the extra branches removed noise, not useful knowledge.

A smaller tree is **more interpretable**: a doctor could read every rule and check that it makes sense. A tree with many leaves is accurate on the training data but hard to explain and less reliable on new patients.

### 8.1 Drawing the pruned tree

The tree pruned with `min_samples_leaf` is small enough to draw completely. Compare it with the full tree drawing in Section 5.1: this one fits on the screen and every rule can be read. Look at the `samples` value in each leaf: every leaf has at least 20 patients, exactly what `min_samples_leaf` asked for.

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(pruned_tree,
          feature_names=list(X.columns),
          class_names=class_names,
          filled=True, rounded=True, fontsize=9)
plt.title("Pruned decision tree (min_samples_leaf)")
plt.show()

### 8.2 Feature importance

A decision tree can tell us **which features helped the most**. A feature's importance is the total reduction in impurity (Gini) it produced across all its splits. All importances add up to 1. A feature with importance 0 was **never used** by the tree.

Let us compare which features the full tree and the pruned tree actually use.

In [ ]:
full_importance = pd.Series(full_tree.feature_importances_, index=X.columns).sort_values(ascending=False)
pruned_importance = pd.Series(pruned_tree.feature_importances_, index=X.columns).sort_values(ascending=False)

full_used = full_importance[full_importance > 0]        # keep only features the tree used
pruned_used = pruned_importance[pruned_importance > 0]

print("Full tree uses  ", len(full_used), "of", X.shape[1], "features")
print("Pruned tree uses", len(pruned_used), "of", X.shape[1], "features")
print()
print("Features used by the pruned tree:")
print(pruned_used.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(full_used.index, full_used.values, color="gray")
axes[0].invert_yaxis()                 # biggest bar at the top
axes[0].set_xlabel("Importance")
axes[0].set_title("Full tree")
axes[0].set_xlim(0, 1)

axes[1].barh(pruned_used.index, pruned_used.values, color="steelblue")
axes[1].invert_yaxis()
axes[1].set_xlabel("Importance")
axes[1].set_title("Pruned tree")
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

**What do we see?**

- The full tree uses many features, but most of them have very small importance. Those small splits mostly fit noise in the training data.
- The pruned tree makes its decision with only a **few** features, and it is **more accurate** on the test set.
- One feature (`worst concave points`) does most of the work in both trees.

This is a key strength of decision trees: we can see exactly which measurements drive the prediction.

## 9. Gini vs Entropy on the Real Data

The `criterion` setting chooses which impurity measure the tree uses. Let us compare both with the same depth.

In [ ]:
for criterion in ["gini", "entropy"]:
    model = DecisionTreeClassifier(criterion=criterion, max_depth=best_depth, random_state=42)
    model.fit(X_train, y_train)

    root_feature = X.columns[model.tree_.feature[0]]     # the feature used at the root node
    val_acc = accuracy_score(y_val, model.predict(X_val))

    print(f"criterion = {criterion:8s} | validation accuracy = {val_acc:.3f} | root feature = {root_feature}")

In practice, Gini and entropy usually give **very similar trees and accuracy**. Gini is slightly faster to compute (no logarithm), which is why it is the default in scikit-learn.

`model.tree_.feature[0]` gives the column number used at the root node, and `X.columns[...]` turns it into the feature name. You will use this in the exercises.

## 10. Summary

| Idea | What to remember |
|---|---|
| Attribute selection | At each node, choose the attribute with the **highest information gain** (largest drop in Gini or entropy). |
| Gini vs entropy | Both measure impurity. They usually choose similar splits. Gini is the default. |
| Full tree | Training accuracy 100%, but it **overfits** and is hard to read. |
| `max_depth` | Too small leads to underfitting, too large leads to overfitting. Choose it with the validation set. |
| Pre-pruning | Stop growth early with `max_depth`, `min_samples_leaf`, `min_samples_split` or `max_leaf_nodes`. Choose the value with the validation set; for a tie, take the simpler tree. |
| Interpretability | A smaller tree is easier to explain and often generalises **just as well or better**. |
| Validation vs test | Choose settings with the **validation** set. Use the **test** set only once at the end. |

---
# PART B - Graded Exercises (100 marks, about 30 minutes)

### Instructions

- Work **on your own**.
- Fill in every place marked `____` (four underscores) and every cell marked **Your answer:**.
- Do **not** change the `random_state` values. Everyone must get the same numbers so your work can be marked.
- Run every cell from top to bottom before you submit. Code that does not run cannot receive full marks.
- Save the notebook as `AIML202_DecisionTree_<YourName>.ipynb` and submit it.

| Exercise | Topic | Marks |
|---|---|---|
| Exercise 1 | Attribute selection | 30 |
| Exercise 2 | Tree depth | 35 |
| Exercise 3 | Pre-pruning and interpretability | 35 |
| **Total** | | **100** |

### The dataset: Wine

You will use the **Wine** dataset from scikit-learn: 178 wines from **3 different producers** (classes `class_0`, `class_1`, `class_2`), with **13 chemical features** such as `alcohol`, `proline` and `color_intensity`.

Run the setup cell below first. It loads the data, splits it, and defines the helper functions you need.

In [2]:
# ===== SETUP: run this cell, do not change it =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

wine = load_wine()
Xw = pd.DataFrame(wine.data, columns=wine.feature_names)
yw = wine.target
wine_classes = wine.target_names.tolist()

Xw_temp, Xw_test, yw_temp, yw_test = train_test_split(Xw, yw, test_size=0.2, random_state=53, stratify=yw)
Xw_train, Xw_val, yw_train, yw_val = train_test_split(Xw_temp, yw_temp, test_size=0.25, random_state=53, stratify=yw_temp)

print("Training:", len(Xw_train), " Validation:", len(Xw_val), " Test:", len(Xw_test))


def gini(yes, no):
    total = yes + no
    p_yes = yes / total
    p_no = no / total
    return 1 - (p_yes ** 2 + p_no ** 2)


def entropy(yes, no):
    total = yes + no
    result = 0
    for count in [yes, no]:
        p = count / total
        if p > 0:
            result = result - p * np.log2(p)
    return result

Training: 106  Validation: 36  Test: 36


## Exercise 1 - Attribute Selection (30 marks)

### Part 1a - Calculate by hand with code (15 marks)

A node contains **20 customers**: **12 "Yes"** (bought the product) and **8 "No"**.

We can split this node with one of two attributes:

| Split | Left group | Right group |
|---|---|---|
| **Attribute A** | 10 Yes, 2 No (12 customers) | 2 Yes, 6 No (8 customers) |
| **Attribute B** | 6 Yes, 4 No (10 customers) | 6 Yes, 4 No (10 customers) |

Complete the code to compute the **Gini gain** of each split. Use the `gini` function from the setup cell.

Remember: weight of a group = (customers in the group) / (customers in the node).

In [16]:
# Gini of the node before the split
gini_before = gini(8, 4)  # 12 total samples: 8 Yes, 4 No (update based on your dataset counts)

# Attribute A
gini_A_left = gini(10, 2)
gini_A_right = gini(2, 6)
weighted_A = (12 / 20) * gini_A_left + (8 / 20) * gini_A_right
gain_A = gini_before - weighted_A

# Attribute B
gini_B_left = gini(6, 4)
gini_B_right = gini(6, 4)
weighted_B = (10 / 20) * gini_B_left + (10 / 20) * gini_B_right
gain_B = gini_before - weighted_B

print("Gini before split:", round(gini_before, 3))
print("Gain of Attribute A:", round(gain_A, 3))
print("Gain of Attribute B:", round(gain_B, 3))

Gini before split: 0.444
Gain of Attribute A: 0.128
Gain of Attribute B: -0.036


**Question 1a (5 marks):** Which attribute should the tree choose? Explain in one or two sentences **why** Attribute B gives (almost) no gain. Hint: compare the Yes/No mix in its groups with the mix in the original node.

**Your answer:**

### Part 1b - Gini vs entropy on the Wine data (15 marks)

Train two trees on the Wine training data, both with `max_depth=3` and `random_state=42`:
one with `criterion="gini"` and one with `criterion="entropy"`.
For each tree, print its **validation accuracy** and the **feature used at the root**.

In [17]:
for criterion in ["gini", "entropy"]:
    model = DecisionTreeClassifier(criterion=criterion, max_depth=3, random_state=42)  # (3 marks)
    model.fit(Xw_train, yw_train)                                                      # (3 marks)

    val_acc = accuracy_score(yw_val, model.predict(Xw_val))                            # (3 marks)
    root_feature = Xw.columns[model.tree_.feature[0]]

    print(f"criterion = {criterion:8s} | validation accuracy = {val_acc:.3f} | root feature = {root_feature}")

criterion = gini     | validation accuracy = 0.833 | root feature = proline
criterion = entropy  | validation accuracy = 0.889 | root feature = proline


**Question 1b (6 marks):** Did the two criteria choose the same root feature? Was there a big difference in validation accuracy? What does this tell you about Gini vs entropy?

**Your answer:**

## Exercise 2 - Effect of Tree Depth (35 marks)

### Part 2a - Train trees of different depths (15 marks)

Complete the loop to train a tree for every `max_depth` from 1 to 8 (use `random_state=42`) and save the training and validation accuracy in the two lists.

In [18]:
for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)  # (3 marks)
    model.fit(Xw_train, yw_train)                               # (3 marks)

    train_acc = accuracy_score(yw_train, model.predict(Xw_train)) # (3 marks)
    val_acc = accuracy_score(yw_val, model.predict(Xw_val))      # (3 marks)

    train_acc_list.append(train_acc)                            # (3 marks)
    val_acc_list.append(val_acc)                                # (3 marks)

NameError: name 'depths' is not defined

### Part 2b - Plot the results (10 marks)

Complete the plot: draw both lines, and add the axis labels and a title.

In [19]:
plt.figure(figsize=(8, 5))
plt.plot(depths, train_acc_list, marker="o", label="Training accuracy")   # (2 marks)
plt.plot(depths, val_acc_list, marker="o", label="Validation accuracy")     # (2 marks)
plt.xlabel("Max Depth")                                                     # (2 marks)
plt.ylabel("Accuracy")                                                      # (2 marks)
plt.title("Training vs Validation Accuracy by Tree Depth")                  # (2 marks)
plt.legend()
plt.grid(True)
plt.show()

best_depth_wine = list(depths)[int(np.argmax(val_acc_list))]
print("Best max_depth:", best_depth_wine)

NameError: name 'depths' is not defined

<Figure size 800x500 with 0 Axes>

**Question 2c (10 marks):** Look at your table and plot.

1. Which depth is the best, and why did we choose it using the **validation** set and not the training set? (4 marks)
2. At which depth does the training accuracy reach 1.0? Is that tree the best one for new data? Use the words **overfitting** or **underfitting** in your answer. (4 marks)
3. Is a tree with `max_depth=1` overfitting or underfitting? (2 marks)

**Your answer:**

## Exercise 3 - Pre-pruning and Interpretability (35 marks)

### Part 3a - Choosing min_samples_split (12 marks)

Complete the loop to train one tree for each value of `min_samples_split` (use `random_state=42`) and record its accuracy and number of leaves.

In [20]:
full_wine_tree = DecisionTreeClassifier(random_state=42)
full_wine_tree.fit(Xw_train, yw_train)

split_values = [2, 10, 20, 30, 40, 50]
split_train_acc = []
split_val_acc = []
split_n_leaves = []

for v in split_values:
    model = DecisionTreeClassifier(min_samples_split=v, random_state=42)  # (3 marks)
    model.fit(Xw_train, yw_train)

    split_train_acc.append(accuracy_score(yw_train, model.predict(Xw_train)))
    split_val_acc.append(accuracy_score(yw_val, model.predict(Xw_val)))
    split_n_leaves.append(model.get_n_leaves())  # (3 marks)

split_results = pd.DataFrame({
    "min_samples_split": split_values,
    "Train accuracy": np.round(split_train_acc, 3),
    "Validation accuracy": np.round(split_val_acc, 3),
    "Leaves": split_n_leaves,
})
split_results

,min_samples_split,Train accuracy,Validation accuracy,Leaves
0,2,1.000,0.833,8
1,10,1.000,0.833,8
2,20,1.000,0.833,8
3,30,0.972,0.889,6
4,40,0.962,0.889,5
5,50,0.915,0.833,3


The cell below picks the best value for you (highest validation accuracy; for a tie, the largest value, which gives the simpler tree). Complete it to train the final pre-pruned tree and compare it with the full tree **on the test set**.

In [ ]:
best_val = max(split_val_acc)
best_split = 2
for i in range(len(split_values)):
    if split_val_acc[i] == best_val:
        best_split = split_values[i]
print("Chosen min_samples_split:", best_split)

pruned_wine_tree = DecisionTreeClassifier(min_samples_split=____, random_state=42)    # (3 marks)
pruned_wine_tree.fit(Xw_train, yw_train)

for name, model in [["Full tree", full_wine_tree], ["Pruned tree", pruned_wine_tree]]:
    test_acc = accuracy_score(____, model.predict(____))                              # (3 marks)
    print(f"{name:12s} | test accuracy = {test_acc:.3f} | depth = {model.get_depth()} | leaves = {model.get_n_leaves()}")

### Part 3b - Visualize and interpret the pruned tree (13 marks)

Draw the pruned tree and list the features it actually uses (importance greater than 0).

In [21]:
best_val = max(split_val_acc)
best_split = 2
for i in range(len(split_values)):
    if split_val_acc[i] == best_val:
        best_split = split_values[i]
print("Chosen min_samples_split:", best_split)

best_pruned_tree = DecisionTreeClassifier(min_samples_split=best_split, random_state=42)
best_pruned_tree.fit(Xw_train, yw_train)

full_test_acc = accuracy_score(yw_test, full_wine_tree.predict(Xw_test))
pruned_test_acc = accuracy_score(yw_test, best_pruned_tree.predict(Xw_test))

print("Full tree test accuracy:   ", round(full_test_acc, 3))
print("Pruned tree test accuracy:", round(pruned_test_acc, 3))
print("Pruned tree leaves:       ", best_pruned_tree.get_n_leaves())

Chosen min_samples_split: 40
Full tree test accuracy:    0.833
Pruned tree test accuracy: 0.889
Pruned tree leaves:        5


**Question 3b (5 marks):** Using your tree drawing, write **one rule** in plain English that the tree uses to predict a class. Follow one path from the root to a leaf.

Example format: "If [feature] is less than or equal to [value] and [feature] is greater than [value], then the wine is [class]."

**Your answer:**

**Question 3c (10 marks):** Look at your results from Part 3a.

1. How many leaves did pre-pruning remove, and what happened to the test accuracy? (4 marks)
2. In the table, what happens to the tree and its validation accuracy when `min_samples_split` is too large (50)? Is this overfitting or underfitting? (2 marks)
3. Which tree would you give to a wine expert who wants to **understand** how the decision is made? Explain your choice using the words **accuracy** and **interpretability**. (4 marks)

**Your answer:**

---
**End of lab.** Run all cells once more from top to bottom (Kernel menu, then Restart and Run All), save, and submit your notebook.